In [ ]:
from google.colab import drive

drive.mount('/content/drive')

!pip install ultralytics

In [ ]:
!mkdir -p /content/dataset
!unzip -q /content/drive/MyDrive/dataset.zip -d /content/dataset

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from ultralytics import YOLO

In [ ]:
dataset_root = r"./dataset" 
drive_save_path = "/content/drive/MyDrive/Train_YOLO"


def format_duration(total_seconds: float) -> str:
    total_seconds = max(0, int(total_seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


if __name__ == "__main__":
    # Khởi tạo Model và Huấn luyện
    model = YOLO("yolov8l-cls.pt")
    start_time = time.perf_counter()

    results = model.train(
        data=dataset_root,
        epochs=120,
        imgsz=384,
        batch=64,
        optimizer="SGD",
        momentum=0.937,
        patience=20,
        augment=True,
        lr0=0.01,
        lrf=0.01,
        cos_lr=True,

        dropout=0.2,
        fliplr=0.5,
        flipud=0.5,

        degrees=15.0,
        perspective=0.0001,
        shear=2.0,

        mixup=0.15,
        copy_paste=0.1,

        multi_scale=True,

        workers=8,
        project=drive_save_path,
        name="train_model",
        # cache=True,
        device=0,
        deterministic=True,
        pretrained=True,
        exist_ok=True
    )

    val_results = model.val(data=dataset_root, split="val")

    test_results = model.val(data=dataset_root, split="test")

    best_model_path = "/content/drive/MyDrive/Train_YOLO/train_model/weights/best.pt"

    model = YOLO(best_model_path)

    test_dir = os.path.join(dataset_root, "test")
    class_names = sorted(os.listdir(test_dir))

    y_true = []
    y_pred = []

    for class_id, class_name in enumerate(class_names):
        class_path = os.path.join(test_dir, class_name)
        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)

            result = model(img_path, verbose=False)
            pred_class = result[0].probs.top1

            y_true.append(class_id)
            y_pred.append(pred_class)

    print("\n📄 CLASSIFICATION REPORT:")
    report_str = classification_report(y_true, y_pred, target_names=class_names)
    print(report_str)

    # 2. Lưu Classification Report thành ảnh (dạng Heatmap)
    report_dict = classification_report(
        y_true, y_pred, target_names=class_names, output_dict=True
    )
    # Chuyển thành DataFrame và loại bỏ các dòng không cần thiết để vẽ biểu đồ đẹp hơn
    report_df = pd.DataFrame(report_dict).transpose()
    # Loại bỏ cột 'support' và dòng 'accuracy' để chỉ tập trung vào Precision, Recall, F1
    plot_df = report_df.drop(columns=["support"]).iloc[:-3, :]

    plt.figure(figsize=(10, 6))
    sns.heatmap(plot_df, annot=True, cmap="RdYlGn", fmt=".2f", cbar=True)
    plt.title("Classification Report Heatmap")
    plt.savefig(
        "classification_report_result.png", dpi=300, bbox_inches="tight"
    )  # Lưu ảnh chất lượng cao
    plt.show()

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
    )
    plt.xlabel("Dự đoán (Predicted)")
    plt.ylabel("Thực tế (Actual)")
    plt.title("Confusion Matrix - Trash Classification")
    plt.savefig("confusion_matrix_result.png")
    plt.show()

    elapsed_time = time.perf_counter() - start_time
    print(f"⏱️ Tổng thời gian: {format_duration(elapsed_time)}")

In [ ]:
from google.colab import drive
import os
import time

drive.mount('/content/drive')

!pip install ultralytics split-folders

from ultralytics import YOLO

# Cấu hình đường dẫn
DRIVE_PATH = "/content/drive/MyDrive/Train_YOLO"
DATASET_ZIP = "/content/drive/MyDrive/dataset.zip" # Đường dẫn file zip trên Drive của bạn
DATASET_ROOT = "/content/dataset"

last_checkpoint = os.path.join(DRIVE_PATH, "train_model", "weights", "last.pt")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
if not os.path.exists(DATASET_ROOT):
    !mkdir -p {DATASET_ROOT}
    !unzip -q {DATASET_ZIP} -d {DATASET_ROOT}

else:
    print("--- Dataset đã tồn tại trên máy ảo, bỏ qua bước giải nén. ---")

In [ ]:
start_time = time.perf_counter()

if os.path.exists(last_checkpoint):
    print("--- Đang thực hiện Resume thủ công... ---")
    # Load trọng số từ file last.pt
    model = YOLO(last_checkpoint)

    # metrics = model.val()

    # print(metrics)

    results = model.train(
        data=DATASET_ROOT,
        epochs=120,
        imgsz=384,
        batch=64,
        optimizer="SGD",
        momentum=0.937,
        patience=20,
        augment=True,
        lr0=0.0005,
        lrf=0.01,
        cos_lr=False,

        dropout=0.2,
        fliplr=0.5,
        flipud=0.5,

        degrees=15.0,
        perspective=0.0001,
        shear=2.0,

        mixup=0.15,
        copy_paste=0.1,

        multi_scale=True,

        workers=8,
        project=DRIVE_PATH,
        name="train_model",
        device=0,
        deterministic=True,
        pretrained=True,
        exist_ok=True,
    )